# 05 · NLP analysis of feedback
TF-IDF + logistic regression with participant-grouped CV, top terms and exploratory clusters.

> **Prerequisite:** `python -m gazegrounded.cli run-all` (synthetic demo).
>
> All logic lives in `src/gazegrounded/`; this notebook only calls it. With the synthetic config every number is **SYNTHETIC DEMO DATA — NOT RESEARCH EVIDENCE**.

Synthetic comments come from templates, so the classifier scores are trivially high.

In [1]:
from pathlib import Path
import os
# Run from the repository root so relative config paths resolve.
while not Path("pyproject.toml").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
import numpy as np
import pandas as pd
from gazegrounded.config import load_config
from gazegrounded.data.loader import load_processed
from gazegrounded.utils import SYNTHETIC_BANNER
cfg = load_config("configs/synthetic.yaml")  # switch to configs/experiment.yaml for real data
ds = load_processed(cfg)
print(SYNTHETIC_BANNER if ds.is_synthetic else "Real data:", cfg.data.source)

SYNTHETIC DEMO DATA — NOT RESEARCH EVIDENCE synthetic


In [2]:
from gazegrounded.nlp.preprocessing import prepare_feedback
fb = prepare_feedback(ds.trials)
print('with feedback:', int(fb.has_feedback.sum()), '/', len(fb))
fb[['feedback_text','feedback_clean','feedback_category']].dropna().head(8)

with feedback: 113 / 120


,feedback_text,feedback_clean,feedback_category
0,Hard to find. I looked all over the page befor...,hard to find i looked all over the page before...,findability
1,"No issues, I found it immediately.",no issues i found it immediately,no_friction
3,The label didn't make it clear what it does.,the label didn't make it clear what it does,ambiguous_wording
4,"Too much text on the page, the spending limit ...",too much text on the page the spending limit w...,information_overload
5,The spending limit was not where I expected it...,the spending limit was not where i expected it...,findability
6,"Easy, the freeze button was right there.",easy the freeze button was right there,no_friction
7,"No issues, I found it immediately.",no issues i found it immediately,no_friction
8,"Fine, but the page looked busy.",fine but the page looked busy,other


In [3]:
from gazegrounded.utils import latest_run_dir
import json
run = latest_run_dir(cfg.experiments_dir(), 'nlp-feedback')
m = json.loads((run/'metrics.json').read_text())
print('majority-class accuracy', round(m['majority_class_accuracy'], 2))
print({k: round(v, 3) for k, v in m['folds'].items()})
pd.DataFrame(m['out_of_fold']['per_class']).T.round(2)

majority-class accuracy 0.34
{'accuracy_fold_mean': 0.941, 'macro_f1_fold_mean': 0.924, 'weighted_f1_fold_mean': 0.942, 'accuracy_fold_sd': 0.037, 'macro_f1_fold_sd': 0.054, 'weighted_f1_fold_sd': 0.038, 'n_folds': 5}


,precision,recall,f1,support
ambiguous_wording,1.00,1.00,1.00,18.0
findability,0.86,0.75,0.80,16.0
information_overload,0.78,0.88,0.82,8.0
navigation,0.80,1.00,0.89,12.0
no_friction,1.00,0.97,0.99,38.0
other,1.00,1.00,1.00,13.0
visual_hierarchy,1.00,0.88,0.93,8.0


In [4]:
pd.read_csv(run/'feature_importance.csv').groupby('class').head(3)

,class,term,coefficient
0,ambiguous_wording,didn't,0.744340
1,ambiguous_wording,label,0.695149
2,ambiguous_wording,what,0.695149
8,findability,find,0.816058
9,findability,where,0.779589
10,findability,the address,0.650280
16,information_overload,too much,0.903915
17,information_overload,much,0.903915
18,information_overload,too,0.741777
24,navigation,menu,0.940684


In [5]:
pd.read_csv(run/'clusters.csv')

,cluster,size,top_terms
0,0,50,"clear, quick clear, quick, label, didn't, make..."
1,1,13,"no problems, problems, straightforward no, str..."
2,2,7,"click, where click, obvious, obvious where, wh..."
3,3,22,"address, where, option, not, expected, change"
4,4,14,"page, looked, looked busy, fine page, fine, pa..."
5,5,7,"immediately, issues immediately, issues, no is..."
